# LavaLabs Atmospheric Lab — one fixed CatBoost comparison
Private, supplied-data experiment; same retained 83 predictors. Iterative development, not untouched confirmation.
Fixed Ordered CPU recipe, no parameter sweep, 90 fit/two-hour cap. SI-assisted code; dependencies and starter credited.


In [ ]:
from pathlib import Path
import json, os, sys
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
package = Path('/kaggle/working/atmospheric_lab')
package.mkdir(exist_ok=True)
sources = json.loads('{"__init__.py": "\\"\\"\\"LavaLabs Atmospheric Lab. SI-assisted code; scientific inputs retain provenance.\\"\\"\\"\\n", "baseline.py": "\\"\\"\\"First measured baseline. Official HistGB recipe, supplied features, no selection.\\"\\"\\"\\nfrom pathlib import Path\\nimport importlib.metadata\\nimport platform\\nimport time\\nimport numpy as np\\nfrom sklearn.ensemble import HistGradientBoostingRegressor\\nfrom threadpoolctl import threadpool_limits\\nfrom .core import TARGETS, SOURCE, load_tables, site_folds, metric, export_submission, write_json\\n\\nPARAMS = dict(max_iter=250, learning_rate=.04, max_depth=3,\\n              min_samples_leaf=40, l2_regularization=5., max_features=.6,\\n              random_state=0, early_stopping=False)\\n\\n\\ndef fit_predict(x, y, z, params=None):\\n    y = np.asarray(y, dtype=float)\\n    keep = np.isfinite(y)\\n    if keep.sum() < 2:\\n        raise ValueError(\'Not enough measured labels in the fitting partition\')\\n    with threadpool_limits(limits=2):\\n        model = HistGradientBoostingRegressor(**(PARAMS if params is None else params))\\n        model.fit(np.asarray(x)[keep], y[keep])\\n        prediction = model.predict(z)\\n    if not np.isfinite(prediction).all():\\n        raise ValueError(\'Estimator produced a nonfinite prediction\')\\n    return model, prediction\\n\\n\\ndef run(root, output, n_splits=5, params=None, expected_rows=1221):\\n    started = time.monotonic()\\n    output = Path(output)\\n    output.mkdir(parents=True, exist_ok=True)\\n    train, test, sample, names = load_tables(root, expected_rows)\\n    x = train[names].to_numpy(dtype=float)\\n    z = test[names].to_numpy(dtype=float)\\n    y = train[TARGETS].to_numpy(dtype=float)\\n    folds = site_folds(train.site_key.to_numpy(), n_splits)\\n    oof = np.full(y.shape, np.nan)\\n    mean_oof = np.full(y.shape, np.nan)\\n    fold_reports = []\\n    for fold_number, (fit, check) in enumerate(folds, 1):\\n        tick = time.monotonic()\\n        for j in range(5):\\n            _, oof[check, j] = fit_predict(x[fit], y[fit, j], x[check], params)\\n            mean_oof[check, j] = np.nanmean(y[fit, j])\\n        report = metric(y[check], oof[check])\\n        report.update(fold=fold_number, seconds=round(time.monotonic()-tick, 3),\\n                      heldout_sites=int(train.iloc[check].site_key.nunique()),\\n                      fit_sites=int(train.iloc[fit].site_key.nunique()))\\n        fold_reports.append(report)\\n        print(f\'Fold {fold_number}/{n_splits}: MCRMSE {report[\\"mcrmse\\"]:.6f}; {report[\\"seconds\\"]:.1f}s\', flush=True)\\n    predictions = np.zeros((len(test), 5))\\n    for j in range(5):\\n        _, predictions[:, j] = fit_predict(x, y[:, j], z, params)\\n    export_hash = export_submission(sample, predictions, output/\'submission_table.csv\')\\n    receipt = {\'schema\': 1, \'experiment\': \'supplied_table_unselected_histgb\',\\n               \'si_disclosure\': \'OpenAI Codex assisted code, tests and documentation; source observations retain competition provenance.\',\\n               \'baseline_reference\': SOURCE, \'baseline_adaptation\': \'29 supplied features; explicit no early stopping; no OOF-tuned shrinkage.\',\\n               \'counts\': {\'train\': len(train), \'test\': len(test), \'sites\': int(train.site_key.nunique()), \'features\': len(names)},\\n               \'validation\': {\'method\': \'whole-site GroupKFold\', \'folds\': n_splits, \'site_overlap\': False,\\n                              \'selection\': \'none; fixed baseline parameters\', \'pooled_model\': metric(y, oof),\\n                              \'pooled_fold_mean\': metric(y, mean_oof), \'pooled_zero\': metric(y, np.zeros_like(y)),\\n                              \'fold_reports\': fold_reports},\\n               \'parameters\': PARAMS if params is None else params, \'seconds\': round(time.monotonic()-started, 3),\\n               \'environment\': {\'python\': platform.python_version(), **{p: importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'scipy\']}},\\n               \'submission\': {\'rows\': len(test), \'sha256\': export_hash, \'finite\': True, \'private_file\': \'submission_table.csv\'},\\n               \'boundary\': \'Unseen-site validation within training time; private/future-time performance is unknown. No leaderboard score yet.\'}\\n    write_json(output/\'baseline_receipt.json\', receipt)\\n    print(\'BASELINE_COMPLETE\', flush=True)\\n    print(receipt[\'validation\'][\'pooled_model\'], flush=True)\\n    return receipt\\n", "cat_followup.py": "\\"\\"\\"One predeclared Ordered CatBoost comparison. No leaderboard parameter search.\\"\\"\\"\\nfrom pathlib import Path\\nimport importlib.metadata\\nimport time\\nimport numpy as np\\nfrom sklearn.model_selection import GroupShuffleSplit\\nfrom .core import TARGETS,load_tables,site_folds,write_json,export_submission\\nfrom .experiment import feature_matrices,_public_summary\\nfrom .selection import nested_compare,final_predict\\n\\nCAT_PARAMS=dict(loss_function=\'RMSE\',iterations=400,learning_rate=.03,depth=3,\\n    grow_policy=\'SymmetricTree\',boosting_type=\'Ordered\',l2_leaf_reg=10.,\\n    bootstrap_type=\'Bernoulli\',subsample=.8,sampling_frequency=\'PerTree\',rsm=.8,\\n    random_strength=1.,random_seed=2026,nan_mode=\'Min\',boost_from_average=True,\\n    use_best_model=False,task_type=\'CPU\',thread_count=2,verbose=False,allow_writing_files=False)\\n\\n\\ndef fit_cat(x,y,z,params=None):\\n    from catboost import CatBoostRegressor\\n    x,y,z=map(np.asarray,[x,y,z])\\n    if x.ndim!=2 or z.ndim!=2 or y.ndim!=1 or len(x)!=len(y) or x.shape[1]!=z.shape[1]:\\n        raise ValueError(\'CatBoost inputs must be numeric matrices and one target vector\')\\n    if any(a.dtype.kind not in \'fiu\' for a in [x,y,z]) or any(np.isinf(a).any() for a in [x,y,z]):\\n        raise ValueError(\'CatBoost inputs must be numeric and cannot contain infinity\')\\n    keep=np.isfinite(y)\\n    if keep.sum()<2:raise ValueError(\'Not enough observed CatBoost fitting labels\')\\n    x,y,z=x.astype(float),y.astype(float),z.astype(float)\\n    if np.ptp(y[keep])==0:\\n        return None,np.full(len(z),float(y[keep][0]))\\n    model=CatBoostRegressor(**(CAT_PARAMS if params is None else params))\\n    model.fit(x[keep],y[keep])\\n    if model.tree_count_!=(CAT_PARAMS if params is None else params)[\'iterations\']:\\n        raise AssertionError(\'Unexpected CatBoost iteration selection\')\\n    prediction=model.predict(z)\\n    if not np.isfinite(prediction).all():raise ValueError(\'Nonfinite CatBoost prediction\')\\n    return model,prediction\\n\\n\\ndef run(root,output,cache=\'/kaggle/temp/lavalabs_features_v1.npz\'):\\n    started=time.monotonic();output=Path(output);output.mkdir(parents=True,exist_ok=True)\\n    train,test,sample,columns=load_tables(root)\\n    x=train[columns].to_numpy(dtype=float);z=test[columns].to_numpy(dtype=float)\\n    y=train[TARGETS].to_numpy(dtype=float);groups=train.site_key.to_numpy()\\n    matrices,features=feature_matrices(root,[*train.sample_id,*test.sample_id],cache)\\n    all_features={\'table\':x,\'official\':np.column_stack([x,matrices[\'official\'][:len(x)]])}\\n    test_features={\'table\':z,\'official\':np.column_stack([z,matrices[\'official\'][len(x):]])}\\n    all_features[\'cat\']=all_features[\'official\'];test_features[\'cat\']=test_features[\'official\']\\n    dev,_=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(x,groups=groups))\\n    counter={\'fits\':0,\'seconds\':[],\'resolved\':None}\\n    def bounded_fit(a,b,c,params=None):\\n        if counter[\'fits\']>=90 or time.monotonic()-started>7200:\\n            raise RuntimeError(\'Fixed CatBoost experiment exceeded its 90-fit / two-hour cap; retain v2.\')\\n        tick=time.monotonic();model,pred=fit_cat(a,b,c)\\n        counter[\'fits\']+=1;counter[\'seconds\'].append(time.monotonic()-tick)\\n        if model is not None and counter[\'resolved\'] is None:counter[\'resolved\']=model.get_all_params()\\n        if counter[\'fits\'] in [1,2]:\\n            estimated=np.mean(counter[\'seconds\'])*(90-counter[\'fits\'])\\n            print(\'CatBoost timed fit\',counter[\'fits\'],round(counter[\'seconds\'][-1],3),\'seconds; remaining fit projection\',round(estimated,1),flush=True)\\n            if estimated+time.monotonic()-started>7200:\\n                raise RuntimeError(\'Fixed CatBoost fit timing exceeds two-hour cap; retain v2 without retuning.\')\\n        return model,pred\\n    comparison=nested_compare({k:v[dev] for k,v in all_features.items()},y[dev],groups[dev],\\n        site_folds(groups[dev],3,seed=2026),inner_splits=3,fitters={\'cat\':bounded_fit})\\n    control=comparison[\'scores\'][\'official\'][\'mcrmse\'];candidate=comparison[\'scores\'][\'cat\'][\'mcrmse\']\\n    fold_wins=sum(f[\'scores\'][\'cat\'][\'mcrmse\']<f[\'scores\'][\'official\'][\'mcrmse\']-1e-10 for f in comparison[\'folds\'])\\n    retained=\'cat\' if candidate<control-1e-10 and fold_wins>=2 else \'official\'\\n    receipt={\'schema\':1,\'experiment\':\'one_fixed_ordered_catboost\',\'parameters\':CAT_PARAMS,\\n      \'catboost_version\':importlib.metadata.version(\'catboost\'),\'comparison\':_public_summary(comparison),\\n      \'decision\':{\'retained\':retained,\'cat_fold_wins\':fold_wins,\'rule\':\'Cat must improve pooled development MCRMSE by >1e-10 and win at least two of three paired outer folds; otherwise retain v2 official.\'},\\n      \'counts\':{\'train\':len(train),\'test\':len(test),\'sites\':len(set(groups)),\'development_sites\':len(set(groups[dev]))},\\n      \'limitations\':[\'This is iterative development on previously used sites, not new independent confirmation.\',\'The previous confirmation sites are not scored again for this comparison.\',\'Private/future-time transfer remains unknown. No public-score fitting.\'],\\n      \'feature_extraction\':features,\'si_disclosure\':\'OpenAI Codex assisted code, tests and analysis. Original competition observations unchanged.\'}\\n    write_json(output/\'selection_before_refit.json\',receipt)\\n    if retained==\'cat\':\\n        pred,choices=final_predict(all_features,test_features,y,groups,\'cat\',n_splits=5,fitters={\'cat\':bounded_fit})\\n        digest=export_submission(sample,pred,output/\'submission_cat.csv\')\\n        receipt.update(final_choices=choices,submission={\'rows\':len(test),\'sha256\':digest,\'file\':\'submission_cat.csv\'})\\n    receipt.update(cat_fits=counter[\'fits\'],cat_fit_seconds=counter[\'seconds\'],resolved_catboost_parameters=counter[\'resolved\'],seconds=time.monotonic()-started)\\n    write_json(output/\'catboost_receipt.json\',receipt)\\n    print(\'FIXED_CATBOOST_COMPLETE\',{\'retained\':retained,\'control\':control,\'candidate\':candidate,\'fold_wins\':fold_wins,\'seconds\':round(receipt[\'seconds\'],1)},flush=True)\\n    return receipt\\n", "core.py": "\\"\\"\\"Contracts and numerical checks for masked, whole-site atmospheric regression.\\"\\"\\"\\nfrom pathlib import Path\\nimport hashlib\\nimport json\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import GroupKFold\\n\\nTARGETS = [\'t_ch4\', \'t_no2\', \'t_co\', \'t_aai\', \'t_firms\']\\nSOURCE = \'https://www.kaggle.com/code/zelza0/obscura-starter-baseline\'\\n\\n\\ndef metric(truth, prediction):\\n    \\"\\"\\"Official pooled, masked target RMSEs; never ignore a bad prediction.\\"\\"\\"\\n    y, p = np.asarray(truth, dtype=float), np.asarray(prediction, dtype=float)\\n    if y.ndim != 2 or y.shape != p.shape or y.shape[1] != 5:\\n        raise ValueError(\'Expected equally shaped n-by-five targets and predictions\')\\n    if np.isinf(y).any() or not np.isfinite(p).all():\\n        raise ValueError(\'Targets cannot be infinite; every prediction must be finite\')\\n    present = np.isfinite(y)\\n    counts = present.sum(axis=0)\\n    if (counts == 0).any():\\n        raise ValueError(\'All five targets need at least one observed value to report MCRMSE\')\\n    errors = [float(np.sqrt(np.mean((y[present[:, j], j] - p[present[:, j], j]) ** 2)))\\n              for j in range(5)]\\n    return {\'mcrmse\': float(np.mean(errors)), \'rmse\': dict(zip(TARGETS, errors)),\\n            \'observed\': dict(zip(TARGETS, map(int, counts)))}\\n\\n\\ndef validate_tables(train, test, sample, expected_rows=None):\\n    for name, frame in [(\'train\', train), (\'test\', test), (\'sample\', sample)]:\\n        if frame.columns.duplicated().any() or \'sample_id\' not in frame:\\n            raise ValueError(f\'{name}: duplicate columns or missing identifier\')\\n        ids = frame[\'sample_id\']\\n        if ids.isna().any() or not ids.is_unique or not ids.map(lambda x: isinstance(x, str) and bool(x.strip())).all():\\n            raise ValueError(f\'{name}: identifiers must be unique, nonempty strings\')\\n    if set(train.sample_id) & set(test.sample_id):\\n        raise ValueError(\'Training and test sample IDs overlap\')\\n    if list(sample.columns) != [\'sample_id\', *TARGETS]:\\n        raise ValueError(\'Submission template columns do not match the contract\')\\n    if list(test.sample_id) != list(sample.sample_id):\\n        raise ValueError(\'Test and template identifier order differs\')\\n    if expected_rows is not None and len(test) != expected_rows:\\n        raise ValueError(\'Unexpected test row count\')\\n    if \'site_key\' not in train or train.site_key.isna().any():\\n        raise ValueError(\'Training sites must be present\')\\n    features = [c for c in test if c != \'sample_id\']\\n    if any(c in features for c in [\'site_key\', *TARGETS]):\\n        raise ValueError(\'Label or site identifier found among predictors\')\\n    if set(train.columns) != {\'sample_id\', \'site_key\', *TARGETS, *features}:\\n        raise ValueError(\'Training and test feature schemas differ\')\\n    for frame, names in [(train, [*features, *TARGETS]), (test, features)]:\\n        for name in names:\\n            if not pd.api.types.is_numeric_dtype(frame[name]) or np.isinf(frame[name].to_numpy(dtype=float)).any():\\n                raise ValueError(\'Predictors and targets must be numerical, missing or finite\')\\n    if train[TARGETS].notna().sum().eq(0).any():\\n        raise ValueError(\'A training target has no observations\')\\n    return features\\n\\n\\ndef load_tables(root, expected_rows=1221):\\n    root = Path(root)\\n    train = pd.read_csv(root / \'train.csv\', dtype={\'sample_id\': str, \'site_key\': str})\\n    test = pd.read_csv(root / \'test.csv\', dtype={\'sample_id\': str})\\n    sample = pd.read_csv(root / \'sample_submission.csv\', dtype={\'sample_id\': str})\\n    features = validate_tables(train, test, sample, expected_rows)\\n    if len(features) != 29:\\n        raise ValueError(\'Expected the official 29 supplied numerical features\')\\n    return train, test, sample, features\\n\\n\\ndef site_folds(groups, n_splits=5, seed=None):\\n    groups = np.asarray(groups)\\n    if pd.isna(groups).any() or len(set(groups)) < n_splits:\\n        raise ValueError(\'Not enough valid sites for the requested folds\')\\n    splitter = GroupKFold(n_splits=n_splits) if seed is None else GroupKFold(n_splits=n_splits,shuffle=True,random_state=seed)\\n    folds = list(splitter.split(np.zeros(len(groups)), groups=groups))\\n    seen = np.zeros(len(groups), dtype=int)\\n    for fit, check in folds:\\n        if set(groups[fit]) & set(groups[check]):\\n            raise AssertionError(\'Site leakage across a fold\')\\n        seen[check] += 1\\n    if not np.all(seen == 1):\\n        raise AssertionError(\'Every row must be held out exactly once\')\\n    return folds\\n\\n\\ndef export_submission(sample, predictions, destination):\\n    p = np.asarray(predictions, dtype=float)\\n    if p.shape != (len(sample), 5) or not np.isfinite(p).all():\\n        raise ValueError(\'Test predictions must be finite and match all template rows\')\\n    if not sample.sample_id.is_unique or not sample.sample_id.map(lambda x: isinstance(x, str)).all():\\n        raise ValueError(\'Invalid submission identifiers\')\\n    result = sample[[\'sample_id\']].copy()\\n    result[TARGETS] = p\\n    path = Path(destination)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    result.to_csv(path, index=False, float_format=\'%.9g\')\\n    replay = pd.read_csv(path, dtype={\'sample_id\': str})\\n    if list(replay.sample_id) != list(sample.sample_id) or not np.isfinite(replay[TARGETS]).all().all():\\n        raise AssertionError(\'Export round trip failed\')\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef write_json(path, value):\\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\\\n\', encoding=\'utf-8\')\\n", "experiment.py": "\\"\\"\\"Bounded content-feature experiments and site-held-out confirmation on Kaggle.\\"\\"\\"\\nfrom concurrent.futures import ThreadPoolExecutor\\nfrom pathlib import Path\\nimport hashlib\\nimport importlib.metadata\\nimport inspect\\nimport json\\nimport platform\\nimport time\\nimport numpy as np\\nfrom sklearn.model_selection import GroupShuffleSplit\\nfrom . import features as feature_module\\nfrom .features import chip_summary, BAND_ORDER, _spatial\\nfrom .baseline import PARAMS\\nfrom .core import TARGETS, load_tables, site_folds, metric, write_json, export_submission\\nfrom .selection import nested_compare, final_predict\\n\\n\\ndef extract_one(path):\\n    import rasterio\\n    with rasterio.open(path) as source:\\n        chip=source.read()  # Intentionally do not inspect geospatial/file metadata.\\n    official=chip_summary(chip,\'official\')\\n    clear=(chip[3]>0)&~np.isin(chip[12],[3,8,9,10])\\n    sufficient=clear.sum()>=500\\n    if sufficient:\\n        revised=dict(official)\\n        revised.update(q_insufficient_clear=0.,q_clear_count=float(clear.sum()),q_cloud_high_fraction=float((chip[13]>=40).mean()))\\n    else:\\n        revised=chip_summary(chip,\'no_fallback\')\\n    bands={name:chip[index].astype(float)*1e-4 for index,name in enumerate(BAND_ORDER[:12])}\\n    spatial={**revised,**_spatial(bands,clear,bool(sufficient))}\\n    return official,revised,spatial\\n\\n\\ndef feature_matrices(root,ids,cache_path,workers=2):\\n    root,cache_path=Path(root),Path(cache_path)\\n    # Both functions define extraction; changing either must invalidate the cache.\\n    source_hash=hashlib.sha256((inspect.getsource(feature_module)+\'\\\\n\'+inspect.getsource(extract_one)).encode()).hexdigest()\\n    fingerprint=hashlib.sha256((source_hash+\'\\\\n\'+str(root.resolve())+\'\\\\n\'+\'\\\\n\'.join(ids)).encode()).hexdigest()\\n    if cache_path.exists():\\n        with np.load(cache_path,allow_pickle=False) as saved:\\n            if saved[\'fingerprint\'].item()==fingerprint:\\n                summary=json.loads(saved[\'summary\'].item())\\n                expected={\'official\':54,\'no_fallback\':57,\'spatial\':73}\\n                for name,width in expected.items():\\n                    value=saved[name]\\n                    names=summary[\'column_names\'][name]\\n                    if (value.shape!=(len(ids),width) or np.isinf(value).any()\\n                            or len(names)!=width or names!=sorted(set(names))\\n                            or summary[\'columns\'][name]!=width or summary[\'chip_count\']!=len(ids)\\n                            or summary[\'feature_source_sha256\']!=source_hash):\\n                        raise ValueError(\'Private feature cache failed schema/integrity checks; rebuild it.\')\\n                print(\'Verified private feature cache loaded.\',flush=True)\\n                return {name:saved[name].copy() for name in expected},summary\\n    rows={name:[] for name in [\'official\',\'no_fallback\',\'spatial\']}\\n    names={}\\n    started=time.monotonic()\\n    def worker(sid):\\n        try: return extract_one(root/\'images\'/f\'{sid}.tif\')\\n        except Exception:\\n            # Keep row identifiers and original path errors out of notebook outputs.\\n            raise RuntimeError(\'A competition image failed extraction; no fabricated fallback is permitted.\') from None\\n    with ThreadPoolExecutor(max_workers=workers) as pool:\\n        for number,triplet in enumerate(pool.map(worker,ids),1):\\n            for name,features in zip(rows,triplet):\\n                keys=sorted(features)\\n                if name not in names: names[name]=keys\\n                if keys!=names[name]: raise AssertionError(\'Feature schema changed across chips\')\\n                rows[name].append([features[k] for k in keys])\\n            if number%250==0 or number==len(ids):\\n                elapsed=time.monotonic()-started\\n                print(f\'Chips {number}/{len(ids)}; {elapsed:.1f}s; estimated {(len(ids)-number)*elapsed/max(number,1)/60:.1f} min remaining\',flush=True)\\n    matrices={name:np.asarray(value,dtype=np.float64) for name,value in rows.items()}\\n    for value in matrices.values():\\n        if np.isinf(value).any(): raise ValueError(\'Infinite extracted predictor\')\\n    revised=matrices[\'no_fallback\']\\n    summary={\'chip_count\':len(ids),\'columns\':{name:len(keys) for name,keys in names.items()},\\n             \'column_names\':names,\'insufficient_clear_fraction\':float(revised[:,names[\'no_fallback\'].index(\'q_insufficient_clear\')].mean()),\\n             \'seconds\':round(time.monotonic()-started,3),\'feature_source_sha256\':source_hash,\\n             \'boundary\':\'Relative pixels only; no geographic location, ground scale or outside observations.\'}\\n    cache_path.parent.mkdir(parents=True,exist_ok=True)\\n    np.savez_compressed(cache_path,**matrices,fingerprint=np.array(fingerprint),summary=np.array(json.dumps(summary)))\\n    return matrices,summary\\n\\n\\ndef _public_summary(comparison):\\n    return {key:comparison[key] for key in [\'scores\',\'folds\',\'fold_mean_score\',\'raw_scores\']}\\n\\n\\nGATE_TOLERANCE = 1e-10\\n\\n\\ndef selection_plan(scores):\\n    \\"\\"\\"Freeze one comparator and at most one novel challenger on development only.\\"\\"\\"\\n    values={name:float(value[\'mcrmse\']) for name,value in scores.items()}\\n    if set(values)!={\'table\',\'official\',\'no_fallback\',\'spatial\'} or not all(np.isfinite(list(values.values()))):\\n        raise ValueError(\'Need four finite development scores\')\\n    control=\'official\' if values[\'official\']<values[\'table\']-GATE_TOLERANCE else \'table\'\\n    novel=\'spatial\' if values[\'spatial\']<values[\'no_fallback\']-GATE_TOLERANCE else \'no_fallback\'\\n    eligible=values[novel]<values[control]-GATE_TOLERANCE\\n    return {\'control\':control,\'novel\':novel,\'challenger_eligible\':bool(eligible),\\n            \'candidate\':novel if eligible else control,\'tolerance\':GATE_TOLERANCE,\\n            \'gate_protocol\':\'v2: development chooses control/challenger; confirmation only accepts or rejects; custom challenger must beat both controls\'}\\n\\n\\ndef confirm_selection(plan,scores):\\n    \\"\\"\\"No best-of-confirmation search and no switching between novel families.\\"\\"\\"\\n    values={name:float(value[\'mcrmse\']) for name,value in scores.items()}\\n    required={\'table\',\'official\',plan[\'candidate\']}\\n    if not required.issubset(values) or not all(np.isfinite(list(values.values()))):\\n        raise ValueError(\'Missing or nonfinite confirmation score\')\\n    control=plan[\'control\']\\n    reasons=[]\\n    if control==\'official\' and not values[\'official\']<values[\'table\']-GATE_TOLERANCE:\\n        control=\'table\'\\n        reasons.append(\'Development-selected official control failed confirmation against table; retain table.\')\\n    retained=control\\n    candidate=plan[\'candidate\']\\n    if plan[\'challenger_eligible\']:\\n        beats_both=all(values[candidate]<values[name]-GATE_TOLERANCE for name in [\'table\',\'official\'])\\n        if beats_both:\\n            retained=candidate\\n            reasons.append(\'Frozen novel challenger passed both confirmation controls.\')\\n        else:\\n            reasons.append(\'Frozen novel challenger failed at least one control; reject it.\')\\n    else:\\n        reasons.append(\'No novel challenger beat the development-selected control; no novel confirmation search.\')\\n    return retained,{\'fallback_control\':control,\'reasons\':reasons}\\n\\n\\ndef run(root,output,cache=\'/kaggle/temp/lavalabs_features_v1.npz\'):\\n    started=time.monotonic()\\n    output=Path(output)\\n    output.mkdir(parents=True,exist_ok=True)\\n    train,test,sample,columns=load_tables(root)\\n    y=train[TARGETS].to_numpy(dtype=float)\\n    groups=train.site_key.to_numpy()\\n    ids=[*train.sample_id,*test.sample_id]\\n    print(\'Input contract passed:\',{\'train\':len(train),\'test\':len(test),\'sites\':len(set(groups))},flush=True)\\n    matrices,feature_receipt=feature_matrices(root,ids,cache)\\n    tx=train[columns].to_numpy(dtype=float)\\n    zx=test[columns].to_numpy(dtype=float)\\n    train_features={\'table\':tx,**{name:np.column_stack([tx,array[:len(train)]]) for name,array in matrices.items()}}\\n    test_features={\'table\':zx,**{name:np.column_stack([zx,array[len(train):]]) for name,array in matrices.items()}}\\n    # The confirmation sites are excluded from candidate/weight selection below.\\n    dev,confirmation=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=2026).split(tx,groups=groups))\\n    assert set(groups[dev]).isdisjoint(groups[confirmation])\\n    dev_features={name:x[dev] for name,x in train_features.items()}\\n    comparison=nested_compare(dev_features,y[dev],groups[dev],site_folds(groups[dev],3,seed=2026),inner_splits=3)\\n    scores=comparison[\'scores\']\\n    plan=selection_plan(scores)\\n    candidate=plan[\'candidate\']\\n    # Freeze the feature family before examining confirmation labels.\\n    selection={**plan,\'basis\':\'development-only control and single challenger selection; simpler families win numerical ties\',\\n               \'development_sites\':len(set(groups[dev])),\'confirmation_sites\':len(set(groups[confirmation])),\\n               \'development_comparison\':_public_summary(comparison)}\\n    write_json(output/\'selection_before_confirmation.json\',selection)\\n    print(\'Feature family frozen before confirmation:\',candidate,flush=True)\\n    confirmation_scores={}\\n    for name in dict.fromkeys([\'table\',\'official\',candidate]):\\n        pred,_=final_predict({k:v[dev] for k,v in train_features.items()},\\n                             {k:v[confirmation] for k,v in train_features.items()},y[dev],groups[dev],name,n_splits=3)\\n        confirmation_scores[name]=metric(y[confirmation],pred)\\n        print(\'Confirmation\',name,confirmation_scores[name][\'mcrmse\'],flush=True)\\n    # Confirmation is an accept/reject gate, so its selected result is not an untouched test estimate.\\n    retained,gate=confirm_selection(plan,confirmation_scores)\\n    prediction,choices=final_predict(train_features,test_features,y,groups,retained,n_splits=5)\\n    export_hash=export_submission(sample,prediction,output/\'submission_selected.csv\')\\n    official_prediction,official_choices=final_predict(train_features,test_features,y,groups,\'official\',n_splits=5)\\n    official_hash=export_submission(sample,official_prediction,output/\'submission_official_control.csv\')\\n    receipt={\'schema\':2,\'experiment\':\'cloud_spatial_nested_site_selection\',\\n             \'si_disclosure\':\'OpenAI Codex assisted code, tests, analysis implementation and documentation. Competition observations retain original provenance.\',\\n             \'counts\':{\'train\':len(train),\'test\':len(test),\'sites\':len(set(groups))},\\n             \'feature_extraction\':feature_receipt,\'selection\':selection,\'confirmation\':confirmation_scores,\\n             \'retained_variant\':retained,\'confirmation_gate\':gate,\'retained_choices\':choices,\'official_control_choices\':official_choices,\\n             \'parameters\':PARAMS,\'final_fit\':\'all permitted training rows with observed target masks; final weights from whole-site OOF\',\\n             \'environment\':{\'python\':platform.python_version(),**{p:importlib.metadata.version(p) for p in [\'numpy\',\'pandas\',\'scikit-learn\',\'rasterio\',\'scipy\']}},\\n             \'submissions\':{\'selected\':{\'rows\':len(test),\'sha256\':export_hash,\'file\':\'submission_selected.csv\'},\\n                            \'official_control\':{\'rows\':len(test),\'sha256\':official_hash,\'file\':\'submission_official_control.csv\'}},\\n             \'seconds\':round(time.monotonic()-started,3),\\n             \'limitations\':[\'Confirmation is used for an accept/reject gate; its selected result is not an untouched estimate.\',\\n                            \'Gate v2 was fixed during initial image extraction, before any image development/confirmation scores were observed; it also requires novel changes to beat the official control.\',\\n                            \'The earlier raw baseline aggregate evaluated all training sites before this refinement, including confirmation sites.\',\\n                            \'No true forward-time validation is possible with withheld dates. Private-test error remains unknown.\',\\n                            \'Feature-family selection uses development outer scores; those selected scores have model-selection optimism.\']}\\n    write_json(output/\'experiment_receipt.json\',receipt)\\n    print(\'EXPERIMENT_COMPLETE\',flush=True)\\n    print({\'development\':{name:value[\'mcrmse\'] for name,value in scores.items()},\'candidate\':candidate,\'retained\':retained,\\n           \'confirmation\':{name:value[\'mcrmse\'] for name,value in confirmation_scores.items()},\'seconds\':receipt[\'seconds\']},flush=True)\\n    return receipt\\n", "features.py": "\\"\\"\\"Content-only chip summaries; explicit cloud ablations, no geospatial metadata.\\n\\nThe official feature recipe is adapted from Zelzy\'s Apache-2.0 starter.\\nOpenAI Codex assisted the implementation and tests.\\n\\"\\"\\"\\nimport numpy as np\\n\\nBAND_ORDER = [\'B1\',\'B2\',\'B3\',\'B4\',\'B5\',\'B6\',\'B7\',\'B8\',\'B8A\',\'B9\',\'B11\',\'B12\',\'SCL\',\'CLDPRB\']\\nSTAT_BANDS = [\'B2\',\'B3\',\'B4\',\'B8\',\'B11\',\'B12\']\\nSCL_CLASSES = [2,3,4,5,6,7,8,9,10,11]\\nMIN_CLEAR = 500\\n\\n\\ndef _statistics(values, prefix):\\n    if not values.size:\\n        return {f\'{prefix}_{name}\': float(\'nan\') for name in [\'mean\',\'std\',\'p50\',\'p95\',\'p99\',\'max\']}\\n    p50,p95,p99 = np.percentile(values,[50,95,99])\\n    return {f\'{prefix}_{name}\':float(value) for name,value in\\n            zip([\'mean\',\'std\',\'p50\',\'p95\',\'p99\',\'max\'],[values.mean(),values.std(),p50,p95,p99,values.max()])}\\n\\n\\ndef _spatial(bands, mask, sufficient):\\n    names = [f\'s_{band}_{stat}\' for band in [\'B11\',\'B12\'] for stat in [\'centre_mean\',\'outer_mean\',\'centre_minus_outer\']]\\n    names += [\'s_ndvi_centre_minus_outer\',\'s_bsi_centre_minus_outer\',\'s_B12_tail_contrast\',\'s_B11_tail_contrast\',\\n              \'s_B12_centre_std\',\'s_B12_gradient\',\'s_B11_gradient\',\'s_ndvi_gradient\',\'s_centre_clear_fraction\',\'s_outer_clear_fraction\']\\n    if not sufficient:\\n        return dict.fromkeys(names,float(\'nan\'))\\n    height,width = mask.shape\\n    rows,cols = np.indices(mask.shape)\\n    centre = (rows>=height/4)&(rows<3*height/4)&(cols>=width/4)&(cols<3*width/4)\\n    c,edge = mask&centre,mask&~centre\\n    output = {}\\n    for band in [\'B11\',\'B12\']:\\n        a = bands[band]\\n        cm = float(a[c].mean()) if c.sum()>=20 else float(\'nan\')\\n        em = float(a[edge].mean()) if edge.sum()>=20 else float(\'nan\')\\n        output.update({f\'s_{band}_centre_mean\':cm,f\'s_{band}_outer_mean\':em,f\'s_{band}_centre_minus_outer\':cm-em})\\n    ndvi = (bands[\'B8\']-bands[\'B4\'])/(bands[\'B8\']+bands[\'B4\']+1e-6)\\n    bsi = ((bands[\'B11\']+bands[\'B4\'])-(bands[\'B8\']+bands[\'B2\']))/(bands[\'B11\']+bands[\'B4\']+bands[\'B8\']+bands[\'B2\']+1e-6)\\n    for label,a in [(\'ndvi\',ndvi),(\'bsi\',bsi)]:\\n        output[f\'s_{label}_centre_minus_outer\'] = float(a[c].mean()-a[edge].mean()) if min(c.sum(),edge.sum())>=20 else float(\'nan\')\\n    for band in [\'B12\',\'B11\']:\\n        p50,p99 = np.percentile(bands[band][mask],[50,99])\\n        output[f\'s_{band}_tail_contrast\'] = float(p99-p50)\\n    output[\'s_B12_centre_std\'] = float(bands[\'B12\'][c].std()) if c.sum()>=20 else float(\'nan\')\\n    for label,a in [(\'B12\',bands[\'B12\']),(\'B11\',bands[\'B11\']),(\'ndvi\',ndvi)]:\\n        vertical = np.abs(np.diff(a,axis=0))[mask[1:]&mask[:-1]]\\n        horizontal = np.abs(np.diff(a,axis=1))[mask[:,1:]&mask[:,:-1]]\\n        values = np.concatenate([vertical,horizontal])\\n        output[f\'s_{label}_gradient\'] = float(values.mean()) if values.size>=20 else float(\'nan\')\\n    output[\'s_centre_clear_fraction\'] = float(mask[centre].mean())\\n    output[\'s_outer_clear_fraction\'] = float(mask[~centre].mean())\\n    return output\\n\\n\\ndef chip_summary(chip, method=\'official\', spatial=False):\\n    chip = np.asarray(chip)\\n    if chip.ndim!=3 or chip.shape[0]!=14 or min(chip.shape[1:])<8:\\n        raise ValueError(\'Expected fourteen chip layers and at least eight pixels per dimension\')\\n    if method not in {\'official\',\'no_fallback\',\'strict\'}:\\n        raise ValueError(\'Unknown cloud-mask method\')\\n    if not np.isfinite(chip).all():\\n        raise ValueError(\'Unexpected nonfinite source chip\')\\n    scl,cloud = chip[12],chip[13]\\n    has_data = chip[3]>0\\n    clear = has_data&~np.isin(scl,[3,8,9,10])\\n    if method==\'strict\': clear = has_data&np.isin(scl,[4,5,6])&(cloud<20)\\n    sufficient = clear.sum()>=MIN_CLEAR\\n    mask = has_data if method==\'official\' and not sufficient else clear\\n    bands = {name:chip[index].astype(np.float64)*1e-4 for index,name in enumerate(BAND_ORDER[:12])}\\n    available = bool(mask.any()) if method==\'official\' else bool(sufficient)\\n    output = {}\\n    for band in STAT_BANDS:\\n        values = bands[band][mask] if available else np.array([])\\n        if method==\'official\' and not values.size: values=np.zeros(1)\\n        output.update(_statistics(values,f\'c_{band}\'))\\n    if available:\\n        b4,b8,b11,b12 = (bands[name][mask] for name in [\'B4\',\'B8\',\'B11\',\'B12\'])\\n        ndvi = (b8-b4)/(b8+b4+1e-6)\\n        values = [ndvi.std(),np.percentile(ndvi,95),np.percentile(b12/(b11+1e-6),95),\\n                  (b12>.2).mean(),(b12>.4).mean(),(b11>.4).mean()]\\n    else:\\n        values = [0.]*6 if method==\'official\' else [float(\'nan\')]*6\\n    output.update(dict(zip([\'c_ndvi_std\',\'c_ndvi_p95\',\'c_b12_b11_p95\',\'c_hot12_gt_0.2\',\'c_hot12_gt_0.4\',\'c_hot11_gt_0.4\'],map(float,values))))\\n    for cls in SCL_CLASSES: output[f\'c_scl_{cls}\']=float((scl==cls).mean())\\n    output[\'c_cloudprob\']=float(cloud.mean())\\n    output[\'c_valid_frac\']=float(mask.mean())\\n    if method!=\'official\':\\n        output[\'q_insufficient_clear\']=float(not sufficient)\\n        output[\'q_clear_count\']=float(clear.sum())\\n        output[\'q_cloud_high_fraction\']=float((cloud>=40).mean())\\n    if spatial: output.update(_spatial(bands,mask,available))\\n    return output\\n", "selection.py": "\\"\\"\\"Target-wise conservative mixtures with selection confined to inner sites.\\"\\"\\"\\nimport time\\nimport numpy as np\\nfrom .core import TARGETS, metric, site_folds\\nfrom .baseline import fit_predict\\n\\nAMPLITUDES = [0.,.25,.5,.75,1.]\\nIMAGE_WEIGHTS = [0.,.5,1.]\\n\\n\\ndef mix_predictions(base, enhanced, mean, amplitude, image_weight):\\n    if not (0<=amplitude<=1 and 0<=image_weight<=1):\\n        raise ValueError(\'Only convex prediction mixtures are permitted\')\\n    base,enhanced,mean = map(np.asarray,[base,enhanced,mean])\\n    return mean+amplitude*((1-image_weight)*(base-mean)+image_weight*(enhanced-mean))\\n\\n\\ndef choose_mix(truth, base, enhanced, means, image_weights=None):\\n    y,b,e,m = map(lambda a:np.asarray(a,dtype=float),[truth,base,enhanced,means])\\n    if any(a.shape!=y.shape for a in [b,e,m]) or not all(np.isfinite(a).all() for a in [b,e,m]):\\n        raise ValueError(\'Mixture inputs must have matching shapes and finite predictions\')\\n    keep = np.isfinite(y)\\n    if np.isinf(y).any() or keep.sum()<2:\\n        raise ValueError(\'Need at least two measured, finite tuning labels\')\\n    best=None\\n    for w in (IMAGE_WEIGHTS if image_weights is None else image_weights):\\n        for a in AMPLITUDES:\\n            p=mix_predictions(b,e,m,a,w)\\n            error=float(np.sqrt(np.mean((y[keep]-p[keep])**2)))\\n            key=(error,w,a)\\n            if best is None or error<best[0][0]-1e-12 or (abs(error-best[0][0])<=1e-12 and (w,a)<best[0][1:]):\\n                best=(key,{\'amplitude\':a,\'image_weight\':w,\'inner_rmse\':error})\\n    return best[1]\\n\\n\\ndef raw_oof(x,y,groups,n_splits=3,params=None,fitter=None):\\n    fitter=fit_predict if fitter is None else fitter\\n    y=np.asarray(y,dtype=float)\\n    predictions=np.full(y.shape,np.nan)\\n    means=np.full(y.shape,np.nan)\\n    for fit,check in site_folds(groups,n_splits):\\n        for j in range(5):\\n            keep=np.isfinite(y[fit,j])\\n            if len(set(np.asarray(groups)[fit][keep]))<2:\\n                raise ValueError(\'Need at least two label-bearing fitting sites per target\')\\n            _,predictions[check,j]=fitter(x[fit],y[fit,j],x[check],params)\\n            means[check,j]=np.nanmean(y[fit,j])\\n    if not np.isfinite(predictions).all() or not np.isfinite(means).all():\\n        raise AssertionError(\'Incomplete inner cross-validation\')\\n    return predictions,means\\n\\n\\ndef nested_compare(features,truth,groups,folds=None,inner_splits=3,params=None,fitters=None):\\n    y=np.asarray(truth,dtype=float)\\n    groups=np.asarray(groups)\\n    features={key:np.asarray(value,dtype=float) for key,value in features.items()}\\n    if \'table\' not in features or any(len(value)!=len(y) for value in features.values()):\\n        raise ValueError(\'Features require the shared table reference and matching rows\')\\n    folds=site_folds(groups,3) if folds is None else folds\\n    predictions={key:np.full(y.shape,np.nan) for key in features}\\n    raw={key:np.full(y.shape,np.nan) for key in features}\\n    means=np.full(y.shape,np.nan)\\n    reports=[]\\n    fitters={} if fitters is None else fitters\\n    seen=np.zeros(len(y),dtype=int)\\n    for number,(fit,check) in enumerate(folds,1):\\n        tick=time.monotonic()\\n        if set(groups[fit])&set(groups[check]) or np.intersect1d(fit,check).size:\\n            raise ValueError(\'Outer facility leakage\')\\n        seen[check]+=1\\n        inner={}\\n        for name,x in features.items():\\n            inner[name],inner_means=raw_oof(x[fit],y[fit],groups[fit],inner_splits,params,fitters.get(name))\\n        for name,x in features.items():\\n            for j in range(5):\\n                _,raw[name][check,j]=fitters.get(name,fit_predict)(x[fit],y[fit,j],x[check],params)\\n        means[check]=np.nanmean(y[fit],axis=0)\\n        choices={}\\n        for name in features:\\n            choices[name]=[]\\n            for j in range(5):\\n                choice=choose_mix(y[fit,j],inner[\'table\'][:,j],inner[name][:,j],inner_means[:,j],\\n                                  image_weights=[0.] if name==\'table\' else None)\\n                choices[name].append(choice)\\n                predictions[name][check,j]=mix_predictions(raw[\'table\'][check,j],raw[name][check,j],\\n                    means[check,j],choice[\'amplitude\'],choice[\'image_weight\'])\\n        reports.append({\'fold\':number,\'fit_sites\':len(set(groups[fit])),\'check_sites\':len(set(groups[check])),\\n                        \'choices\':choices,\'scores\':{name:metric(y[check],p[check]) for name,p in predictions.items()},\\n                        \'seconds\':round(time.monotonic()-tick,3)})\\n        print(\'Outer fold\',number,\'scores:\',{name:round(value[\'mcrmse\'],6) for name,value in reports[-1][\'scores\'].items()},flush=True)\\n    if not np.all(seen==1):\\n        raise ValueError(\'Outer folds must hold every row out exactly once\')\\n    scores={name:metric(y,p) for name,p in predictions.items()}\\n    return {\'scores\':scores,\'folds\':reports,\'predictions\':predictions,\'raw_predictions\':raw,\\n            \'mean_predictions\':means,\'fold_mean_score\':metric(y,means),\\n            \'raw_scores\':{name:metric(y,p) for name,p in raw.items()}}\\n\\n\\ndef final_predict(train_features,test_features,truth,groups,variant,n_splits=5,params=None,fitters=None):\\n    y=np.asarray(truth,dtype=float)\\n    names=[\'table\'] if variant==\'table\' else [\'table\',variant]\\n    oof={}\\n    test_predictions={}\\n    fitters={} if fitters is None else fitters\\n    for name in names:\\n        oof[name],means=raw_oof(np.asarray(train_features[name]),y,groups,n_splits,params,fitters.get(name))\\n        test_predictions[name]=np.zeros((len(test_features[name]),5))\\n        for j in range(5):\\n            _,test_predictions[name][:,j]=fitters.get(name,fit_predict)(train_features[name],y[:,j],test_features[name],params)\\n    mu=np.nanmean(y,axis=0)\\n    result=np.zeros_like(test_predictions[\'table\'])\\n    choices=[]\\n    for j in range(5):\\n        choice=choose_mix(y[:,j],oof[\'table\'][:,j],oof[variant][:,j],means[:,j],\\n                          image_weights=[0.] if variant==\'table\' else None)\\n        result[:,j]=mix_predictions(test_predictions[\'table\'][:,j],test_predictions[variant][:,j],\\n                                    mu[j],choice[\'amplitude\'],choice[\'image_weight\'])\\n        choices.append(choice)\\n    if not np.isfinite(result).all(): raise ValueError(\'Nonfinite final prediction\')\\n    return result,choices\\n"}')
for name, text in sources.items():
    (package/name).write_text(text, encoding='utf-8')
for name in list(sys.modules):
    if name == 'atmospheric_lab' or name.startswith('atmospheric_lab.'):
        del sys.modules[name]
print('LavaLabs source ready; aggregate reporting only.')


In [ ]:
import numpy as np
from atmospheric_lab.core import metric, site_folds
y = np.array([[1,2,3,4,5],[np.nan,4,5,6,7]], dtype=float)
assert abs(metric(y,np.zeros_like(y))['mcrmse'] - np.mean([1,np.sqrt(10),np.sqrt(17),np.sqrt(26),np.sqrt(37)])) < 1e-12
groups = np.repeat(np.arange(6),3)
assert all(set(groups[a]).isdisjoint(groups[b]) for a,b in site_folds(groups,3))
print('Masked metric and whole-site split assertions passed.')


In [ ]:
from pathlib import Path
from atmospheric_lab.cat_followup import run
roots = [Path('/kaggle/input/competitions/obscura'),Path('/kaggle/input/obscura')]
data = next((p for p in roots if (p/'train.csv').exists()),None)
if data is None:
    raise RuntimeError('Attach the official OBSCURA competition input before running.')
receipt = run(data, '/kaggle/working/lavalabs_private_catboost')
print('Private CSV ready. Do not publish row-level outputs.')
print('AGGREGATE_RECEIPT_BEGIN')
print(json.dumps(receipt, sort_keys=True))
print('AGGREGATE_RECEIPT_END')
